# Beyond the Box Score: research notebook

This notebook walks through the analysis interactively. The same code runs end to end in
`python -m bbs.pipeline`. The notebook is for reading the reasoning, poking at the data,
and checking results; the pipeline is the source of truth for the published numbers.

**Question.** Which young NBA players will improve next season, and which signals predict
improvement *beyond* what age and regression to the mean already explain?

It needs one run of `python -m bbs.pipeline` first, which downloads the data. Basketball-Reference
data is not redistributed with this repository.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from bbs import analysis, config, features, models, pipeline

processed = ROOT / "data" / "processed" / "player_seasons.parquet"
if processed.exists():
    ps, design, source = pd.read_parquet(processed), config.DEFAULT_DESIGN, "full data"
elif (ROOT / pipeline.SAMPLE_FILE.relative_to(config.ROOT)).exists():
    ps, design, source = pd.read_csv(ROOT / pipeline.SAMPLE_FILE.relative_to(config.ROOT)), pipeline.SAMPLE_DESIGN, "sample"
else:
    raise FileNotFoundError("No data yet. Run `python -m bbs.pipeline` once from the repository root.")
mt = features.build_model_table(ps, design)
hist = mt[mt["season"] < mt["season"].max()]
print(f"Using {source}: {len(ps):,} player-seasons -> {len(mt):,} young-player rows "
      f"({int(hist['observed'].sum()):,} with an observed next season)")

Using full data: 14,569 player-seasons -> 3,285 young-player rows (2,847 with an observed next season)


## 1. The sample

One row is a player aged 24 or younger with at least 500 minutes in base season *t*. The
outcome is Box Plus/Minus in *t+1*, which counts only if the player logs 250+ minutes.
Errors are weighted by next-season minutes because BPM over 300 minutes is mostly noise.

In [2]:
hist[["player", "season", "age", "mp", "bpm", "bpm_next", "mp_next", "observed"]].sample(8, random_state=1)

,player,season,age,mp,bpm,bpm_next,mp_next,observed
3269,Omer Yurtseven,2022,23.0,706.0,-1.0,-3.9,83.0,0
110,Trevor Ariza,2009,23.0,1999.0,2.4,0.5,2629.0,1
2516,Mitchell Robinson,2021,22.0,853.0,1.2,2.2,1848.0,1
530,Michael Carter-Williams,2016,24.0,1649.0,-0.7,-3.2,846.0,1
348,Calvin Booth,2001,24.0,933.0,0.7,-3.0,279.0,1
1117,Jalen Green,2023,20.0,2602.0,-2.1,-0.5,2601.0,1
1358,Talen Horton-Tucker,2025,24.0,724.0,-2.4,NaN,0.0,0
2602,Dario Šarić,2019,24.0,2023.0,-1.6,-0.4,1632.0,1


## 2. How much "improvement" is just regression to the mean?

If one-year BPM changes were pure development, the average change would be the same
whatever a player's starting point. Instead, bad seasons bounce back and great ones
fade, more so for low-minute players whose BPM is noisier. Any "most improved" list
built from raw changes is contaminated by this.

In [3]:
analysis.regression_to_mean(mt).round(2)

,bpm_bin,bin_mid,minutes,mean_delta,ci_low,ci_high,n
0,"(-10, -4]",-4.62,"1,500+ MP",2.03,1.58,2.48,53
1,"(-10, -4]",-5.02,"500-1,499 MP",2.17,1.90,2.45,203
2,"(-4, -2]",-2.76,"1,500+ MP",1.20,1.01,1.38,254
3,"(-4, -2]",-2.88,"500-1,499 MP",1.10,0.92,1.28,436
4,"(-2, 0]",-0.87,"1,500+ MP",0.56,0.41,0.70,484
5,"(-2, 0]",-1.03,"500-1,499 MP",0.38,0.21,0.56,430
6,"(0, 2]",0.90,"1,500+ MP",0.24,0.07,0.41,428
7,"(0, 2]",0.85,"500-1,499 MP",-0.31,-0.60,-0.03,198
8,"(2, 4]",2.98,"1,500+ MP",0.03,-0.21,0.28,197
9,"(2, 4]",2.75,"500-1,499 MP",-0.79,-1.41,-0.17,39


In [4]:
last = int(mt["outcome_season"].max()) - 1
analysis.naive_most_improved(mt, last)

,player,age,mp,bpm,bpm_next,delta_bpm,mp_next
0,Stephon Castle,20.0,2162.0,-3.2,1.9,5.1,2038.0
1,Tidjane Salaün,19.0,1244.0,-5.2,-0.8,4.4,575.0
2,Reed Sheppard,20.0,654.0,-1.7,2.7,4.4,2147.0
3,Victor Wembanyama,21.0,1527.0,6.5,10.7,4.2,1866.0
4,Alex Sarr,19.0,1814.0,-3.3,0.7,4.0,1305.0
5,Keyonte George,21.0,2109.0,-2.7,0.9,3.6,1786.0
6,Moussa Diabaté,23.0,1241.0,-1.9,1.7,3.6,1899.0
7,Josh Green,24.0,1887.0,-4.3,-0.7,3.6,908.0
8,Jett Howard,21.0,701.0,-5.3,-1.8,3.5,695.0
9,Ajay Mitchell,22.0,597.0,-1.6,1.9,3.5,1473.0


Most of the raw top ten started from a negative BPM. Some are real breakouts, but
the list cannot tell them apart from bad luck reversing. Hence the baseline below.

## 3. Age and survivorship

Development slows quickly after 22. The players who fall out of the rotation are not a
random subset: those projected to be worst vanish most often. A model trained only on
survivors sees an over-optimistic low end, so the pipeline re-checks every conclusion
with inverse-probability weighting.

In [5]:
display(analysis.age_curve(hist).round(3))
analysis.attrition_by_projection(mt).round(3)

,age,mean_delta,ci_low,ci_high,n_observed,n_total,dropout_rate
0,19,1.252,0.914,1.591,115,122,0.057
1,20,1.125,0.908,1.341,287,304,0.056
2,21,0.853,0.669,1.038,425,468,0.092
3,22,0.568,0.418,0.718,586,647,0.094
4,23,0.271,0.126,0.416,691,766,0.098
5,24,0.278,0.141,0.415,743,822,0.096


,quintile,n,dropout_rate,mean_projection,mean_mp
0,Q1 (lowest),626,0.204,-2.755,1164.620
1,Q2,626,0.102,-1.394,1342.107
2,Q3,625,0.080,-0.442,1488.805
3,Q4,626,0.037,0.591,1721.856
4,Q5 (highest),626,0.027,2.860,2143.534


## 4. Baselines first

* **Persistence**: next season equals this season.
* **Age curve**: this season plus the average change at this age.
* **Marcel**: the last three seasons weighted 5/4/3 by minutes, regressed toward the
  mean with *k* minutes of prior, plus a fitted age adjustment (after Tom Tango's
  baseball projection system).

Marcel is the bar to clear. It already "knows" age and regression to the mean.

In [6]:
train = hist[(hist["observed"] == 1) & (hist["season"] < design.first_test_season)]
mar = models.Marcel().fit(train)
print(f"Marcel prior strength k = {mar.k:.0f} minutes; age adjustments:",
      {k: round(v, 2) for k, v in mar.age_adj.items()})

Marcel prior strength k = 250 minutes; age adjustments: {20: 1.05, 21: 0.89, 22: 0.76, 23: 0.55, 24: 0.45}


## 5. Rolling-origin evaluation

For each test season *Y*, every model is refit on seasons before *Y* and then predicts *Y+1*.
This is what a front office could have done in real time; there is no random split and
no peeking. (Gradient boosting is left out here for speed; the pipeline includes it.)

In [7]:
tuned = models.tune(mt, design)
factory = lambda: [models.Persistence(), models.AgeCurve(), models.Marcel(),
                   models.SklearnModel("Ridge", features.ALL_FEATURES, "ridge", tuned["ridge_alpha"])]
preds = models.rolling_origin(mt, factory, design)
models.metrics_by_model(preds).round(3)

,model,n,rmse_w,mae_w,rmse,spearman_delta,top10_hit_rate
0,Persistence,1258,1.906,1.492,1.964,NaN,NaN
1,Age curve,1258,1.780,1.407,1.892,0.155,0.359
2,Marcel,1258,1.691,1.335,1.800,0.367,0.500
3,Ridge,1258,1.638,1.285,1.747,0.407,0.619


In [8]:
pd.DataFrame([models.paired_bootstrap(preds, m, "Marcel", n_boot=1000)
              for m in ["Persistence", "Age curve", "Ridge"]]).round(3)

,model,reference,rmse_diff,ci_low,ci_high,p_model_better,n_players
0,Persistence,Marcel,0.214,0.162,0.266,0.000,505
1,Age curve,Marcel,0.089,0.040,0.138,0.001,505
2,Ridge,Marcel,-0.053,-0.084,-0.022,1.000,505


The ridge model's gain over Marcel is small, about 3% of RMSE, but its player-clustered
bootstrap interval excludes zero. The gain also shows up as better *ranking* of who
improves (rank correlation of predicted vs. actual change, and the top-decile hit
rate).

## 6. Which signals carry the edge?

Weighted least squares of (actual - Marcel projection) on standardized features, with
standard errors clustered by player. Each coefficient is extra next-season BPM per
1 SD of the feature. Strongly collinear features (|r| >= 0.75) are dropped here so the
coefficients are interpretable.

In [9]:
sig = analysis.signal_regression(mt)
print(f"n = {sig.attrs['n']:,}, R^2 on Marcel residuals = {sig.attrs['r2']:.3f}")
sig[["feature", "coef_per_sd", "ci_low", "ci_high", "p_value"]].head(12).round(3)

n = 2,847, R^2 on Marcel residuals = 0.057


,feature,coef_per_sd,ci_low,ci_high,p_value
0,ts_rel,-0.205,-0.314,-0.095,0.000
1,ft_pct_shrunk,0.120,0.034,0.206,0.006
2,f_tr_rel,0.118,0.040,0.196,0.003
3,usg_percent,0.115,0.022,0.209,0.016
4,tov_percent,0.109,0.023,0.196,0.013
5,gs_share,0.099,0.027,0.172,0.007
6,bpm_change,0.098,0.008,0.189,0.034
7,ast_percent,0.079,-0.057,0.216,0.255
8,log_draft_pick,-0.079,-0.163,0.006,0.068
9,undrafted,-0.073,-0.140,-0.006,0.032


**Reading it as a scout would.** Holding the projection fixed, players carrying more
offensive load (usage, trips to the line, ball-handling turnovers) and showing touch
(free-throw %) grow more. Players whose BPM was propped up by above-average shooting
efficiency regress more, because shooting is the noisiest part of the box score. These
are predictive associations. Giving a player more usage would not by itself cause
improvement.

## 7. Where it fails

The largest misses are role changes (trades, promotions to a bigger role) and injuries,
which the box score cannot see. Intervals are also too narrow for players whose minutes
collapse the next season.

In [10]:
r = preds[preds["model"] == "Ridge"].assign(resid=lambda d: d["bpm_next"] - d["pred"])
pd.concat([r.nlargest(5, "resid"), r.nsmallest(5, "resid")])[
    ["player", "season", "age", "bpm", "pred", "bpm_next", "mp_next", "resid"]].round(2)

,player,season,age,bpm,pred,bpm_next,mp_next,resid
841,Victor Oladipo,2017,24.0,-0.5,0.62,5.8,2552.0,5.18
318,Giannis Antetokounmpo,2016,21.0,2.1,2.14,7.3,2845.0,5.16
2754,Isaiah Hartenstein,2021,22.0,-0.9,-0.23,4.5,1216.0,4.73
2792,Ja Morant,2021,21.0,-0.4,1.61,6.1,1889.0,4.49
4419,Vit Krejci,2024,23.0,-3.7,-3.72,0.7,1153.0,4.42
1240,Marquese Chriss,2018,20.0,-2.8,-0.72,-7.5,499.0,-6.78
2805,Michael Porter Jr.,2021,22.0,3.1,2.88,-3.5,265.0,-6.38
388,Cameron Payne,2016,21.0,-0.7,0.15,-6.1,462.0,-6.25
1777,Kevon Looney,2019,22.0,0.5,0.21,-5.6,262.0,-5.81
1813,Dennis Smith Jr.,2019,21.0,-1.9,-0.34,-6.1,537.0,-5.76


See `reports/report.md` for the full write-up, the sensitivity analysis, and the
2026-27 forecast.